# 06 · Natasha spots the fake
*A forged ping has a timestamp from the future. Quality rules turn her instinct into code.*

Pattern: evaluate named rules on every row → valid rows to silver, invalid rows to a **quarantine** table
together with the rules they failed. Nothing is silently dropped.

| Concept | Where you meet it |
|---|---|
| Rules as data (a name → SQL condition dictionary) | setup, Step 1 |
| Quarantine instead of delete | Steps 2 and 3 |
| Re-running gold on clean data | Step 4 |
| A CHECK constraint as the last line of defence | Step 5 |

Deep dive: [docs/06-natasha-spots-the-fake.md](../../docs/06-natasha-spots-the-fake.md)

**Needs:** chapter 03 ran completely (`shield.bronze.incident_reports` with all five files, `shield.bronze.locations`). The notebook imports the rules from
[`src/shield_data/quality.py`](../../src/shield_data/quality.py), so open it from a Git folder in the workspace (see chapter 00).
Chapter 05 is **not** required. This notebook replaces `shield.silver.incident_reports` and rebuilds the gold tables from it.
The **Expected** numbers assume the dataset from chapter 02 (10,000 incidents, seed 42, 5 % dirty); other data gives other numbers and a ❌, which is not a bug.

## Setup

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..", "src")))
from pyspark.sql import Window
from pyspark.sql import functions as F

from shield_data.quality import INCIDENT_RULES


def show_checks(checks):
    """Show (check, result, expected) rows with a ✅ / ❌ for whether result matches expected."""
    fmt = lambda v: f"{v:,}" if isinstance(v, int) else str(v)
    rows = [(what, fmt(got), fmt(want), "✅" if got == want else "❌") for what, got, want in checks]
    display(spark.createDataFrame(rows, "check string, result string, expected string, ok string"))


needed = ["shield.bronze.incident_reports", "shield.bronze.locations"]
missing = [t for t in needed if not spark.catalog.tableExists(t)]
assert not missing, f"Run notebook 03 first. Missing tables: {missing}"

display(spark.createDataFrame(list(INCIDENT_RULES.items()), "rule string, sql_condition string"))

## 1 · Evaluate the rules

### Step 1 · Which rows fail which rules?
**Goal:** tag every row with the list of rules it failed, without dropping anything yet.

| Input | Value |
|---|---|
| `shield.bronze.incident_reports` | 10,250 rows |
| rules | the four above; a row is valid when all conditions are true |

**Expected:** 9,739 valid rows and 511 invalid rows. The 511 include 11 replayed duplicates of bad incidents.
A `NULL` condition counts as a failure (`coalesce(..., false)`).

The sample below lists five invalid rows, for example one with `failed_rules = [valid_severity]` and `severity = catastrophic`.

In [ ]:
bronze_incidents = spark.table("shield.bronze.incident_reports")
parsed = bronze_incidents.withColumn("reported_ts", F.expr("try_to_timestamp(reported_at, \"yyyy-MM-dd'T'HH:mm:ss'Z'\")"))

# One array column with the names of the rules this row FAILED. coalesce: a NULL result counts as a failure.
failed = F.array(
    *[
        F.when(F.coalesce(F.expr(condition), F.lit(False)), F.lit(None)).otherwise(F.lit(name))
        for name, condition in INCIDENT_RULES.items()
    ]
)
checked = parsed.withColumn("failed_rules", F.filter(failed, lambda rule: rule.isNotNull()))

valid_rows = checked.filter(F.size("failed_rules") == 0).drop("failed_rules")
invalid_rows = checked.filter(F.size("failed_rules") > 0)

show_checks([
    ("valid rows", valid_rows.count(), 9_739),
    ("invalid rows", invalid_rows.count(), 511),
])
display(invalid_rows.select("incident_id", "agent_id", "reported_at", "severity", "casualties", "failed_rules").limit(5))

## 2 · Silver and quarantine

### Step 2 · Split the data
**Goal:** valid rows (deduplicated) go to silver, invalid rows go to quarantine with a timestamp. Together they must cover every incident.

**Expected:**

| table | rows |
|---|---|
| `shield.silver.incident_reports` | 9,500 (one per valid incident) |
| `shield.silver.incident_reports_quarantine` | 511 |

Silver (9,500) plus the distinct incidents in quarantine (500) is 10,000, so no incident was lost. Both tables are replaced on every run.

In [ ]:
latest_first = Window.partitionBy("incident_id").orderBy(F.col("_ingested_at").desc())
silver_incidents = valid_rows.withColumn("rn", F.row_number().over(latest_first)).filter("rn = 1").drop("rn")
silver_incidents.writeTo("shield.silver.incident_reports").createOrReplace()

quarantine = invalid_rows.withColumn("_quarantined_at", F.current_timestamp())
quarantine.writeTo("shield.silver.incident_reports_quarantine").createOrReplace()

silver_table = spark.table("shield.silver.incident_reports")
quarantine_table = spark.table("shield.silver.incident_reports_quarantine")
show_checks([
    ("silver rows", silver_table.count(), 9_500),
    ("silver distinct incident_id", silver_table.select("incident_id").distinct().count(), 9_500),
    ("quarantine rows", quarantine_table.count(), 511),
    (
        "silver + quarantine incidents vs bronze incidents",
        silver_table.select("incident_id").union(quarantine_table.select("incident_id")).distinct().count(),
        bronze_incidents.select("incident_id").distinct().count(),
    ),
])

### Step 3 · Which rules fail most?
**Goal:** answer *what kind of garbage are we receiving?* from the quarantine table alone.

**Expected:** every defective row fails exactly one rule, so the counts add up to 511:

| rule | rows |
|---|---|
| `valid_timestamp` | 209 |
| `valid_severity` | 110 |
| `non_negative_casualties` | 96 |
| `has_agent` | 96 |

In [ ]:
failures_by_rule = (
    spark.table("shield.silver.incident_reports_quarantine")
    .select(F.explode("failed_rules").alias("rule"))
    .groupBy("rule")
    .count()
    .orderBy(F.desc("count"))
)
display(failures_by_rule)

counts = {row["rule"]: row["count"] for row in failures_by_rule.collect()}
show_checks([
    ("valid_timestamp", counts.get("valid_timestamp"), 209),
    ("valid_severity", counts.get("valid_severity"), 110),
    ("non_negative_casualties", counts.get("non_negative_casualties"), 96),
    ("has_agent", counts.get("has_agent"), 96),
])

## 3 · Gold on clean data

### Step 4 · Rebuild gold
**Goal:** see the same gold tables as in chapter 05, now computed from silver without the bad rows.

**Expected:**

| check | chapter 05 | now |
|---|---|---|
| incidents in `gold.location_risk` | 10,000 | 9,500 |
| incidents in `gold.daily_threat_briefing` | 9,903 | 9,500 |
| severities in silver | 5 (with `catastrophic`) | 4 |

This step also recreates `silver.locations` from bronze, so it runs without chapter 05.

In [ ]:
spark.sql(
    """
    CREATE OR REPLACE TABLE shield.silver.locations AS
    SELECT location_id, city, country, lat, lon FROM shield.bronze.locations
    """
)
spark.sql(
    """
    CREATE OR REPLACE TABLE shield.gold.daily_threat_briefing AS
    SELECT
      CAST(i.reported_ts AS DATE) AS report_date,
      l.city,
      l.country,
      i.incident_type,
      i.severity,
      COUNT(*)            AS incidents,
      SUM(i.casualties)   AS casualties
    FROM shield.silver.incident_reports i
    JOIN shield.silver.locations l USING (location_id)
    WHERE i.reported_ts IS NOT NULL
    GROUP BY ALL
    """
)
spark.sql(
    """
    CREATE OR REPLACE TABLE shield.gold.location_risk AS
    SELECT
      l.city,
      COUNT(*) AS incidents,
      SUM(CASE WHEN i.severity IN ('high','critical') THEN 1 ELSE 0 END) AS serious_incidents,
      ROUND(AVG(i.casualties), 2) AS avg_casualties
    FROM shield.silver.incident_reports i
    JOIN shield.silver.locations l USING (location_id)
    GROUP BY l.city
    """
)

display(spark.table("shield.silver.incident_reports").groupBy("severity").count().orderBy(F.desc("count")))
show_checks([
    ("incidents in location_risk", spark.table("shield.gold.location_risk").agg(F.sum("incidents")).first()[0], 9_500),
    ("incidents in daily_threat_briefing", spark.table("shield.gold.daily_threat_briefing").agg(F.sum("incidents")).first()[0], 9_500),
    ("distinct severities in silver", spark.table("shield.silver.incident_reports").select("severity").distinct().count(), 4),
])

## 4 · The last line of defence

### Step 5 · A CHECK constraint
**Goal:** rules in the pipeline catch bad data on the way in. A constraint makes the *table* refuse it, whoever writes.

**Expected:**
1. `ADD CONSTRAINT` succeeds, because silver has no negative casualties (Delta checks the existing rows first).
2. The `INSERT` of an incident with `casualties = -1` fails and mentions the constraint `non_negative_casualties`. The error text is not verified yet.
3. Silver still has 9,500 rows.

The first statement drops the constraint if it exists, so the cell can be re-run. Without it a second run fails with `DELTA_CONSTRAINT_ALREADY_EXISTS`.

In [ ]:
spark.sql("ALTER TABLE shield.silver.incident_reports DROP CONSTRAINT IF EXISTS non_negative_casualties")
spark.sql("ALTER TABLE shield.silver.incident_reports ADD CONSTRAINT non_negative_casualties CHECK (casualties >= 0)")

rejected = False
try:
    spark.sql("INSERT INTO shield.silver.incident_reports (incident_id, casualties) VALUES ('BAD', -1)")
except Exception as error:  # noqa: BLE001
    rejected = True
    print("Rejected by the table:", type(error).__name__, "-", str(error).splitlines()[0])

show_checks([
    ("bad insert was rejected", rejected, True),
    ("silver rows unchanged", spark.table("shield.silver.incident_reports").count(), 9_500),
])

## 👀 In the workspace

* **Catalog** → `shield` → `silver` → `incident_reports_quarantine`: sample data shows the `failed_rules` array next to the bad value.
* The constraint is a table property. Run `SHOW TBLPROPERTIES shield.silver.incident_reports` in the SQL editor and look for `non_negative_casualties`.

## 🧹 Clean up

Nothing is deleted: chapter 07 time-travels on `shield.silver.incident_reports` and chapters 10 and 11 reuse it and the gold tables.
The constraint stays on the table on purpose. Re-running chapter 05 replaces the table with `CREATE OR REPLACE`, which should remove it again (not verified in a workspace).

## 🎯 Recap

* Rules are data: one name → condition dictionary, reused by the notebook, the pipeline and the tests.
* Bad rows go to quarantine with the names of the rules they failed: 10,250 bronze rows became 9,500 silver incidents and 511 quarantined rows.
* A CHECK constraint is the last line of defence: the table itself rejects bad writes.

Mini-projects: [docs/06-natasha-spots-the-fake.md](../../docs/06-natasha-spots-the-fake.md#try-it-yourself)

> 💡 Next: chapter 07 goes back in time on this table.